# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [19]:
!py -m pip install langchain_community langchain_experimental langchain_openai


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [21]:
!py -m pip install faiss-cpu 
!py -mpip install google-search-results 


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [27]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
os.environ["SERPAPI_API_KEY"] = "1d0478d7214005b2f8da6b51db4e4b574f6743c79730998bc976b3b036acbab0"

In [28]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [29]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [34]:
from langchain_openai import ChatOpenAI

In [35]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [36]:
OBJECTIVE = "Write a weather report for SF today"

In [37]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [38]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Question: What is the weather report for San Francisco today?
Thought: I need to find the current weather conditions for San Francisco.
Action: Search
Action Input: "current weather San Francisco October 2023"
Observation: ['KRON4 Meteorologist Brady Brewster has the latest Bay Area forecast. SF Bay Area weather forecast Monday, Oct. 6, 2025', 'October Day +82° °. Average values in October Wind speed 15.4 mph Pressure 30 inHg Humidity 57 % Average daytime values for October 2020.', 'Light rainfall is possible across the San Francisco Bay Area on Monday and Tuesday, with accumulations of less than a quarter inch.', 'The early October forecast calls for warm conditions in San Francisco with a slight chance of rain. One detailed outlook lists Thursday, Oct. 1 at about 87°F ...', 'View Day High & Low High 90°F Low 61°F Daily. Condition 11:56 PM 73 °F 54 °F 9.95 in 0 in Fair 10:56 PM 74 °F 58 °F', 'October in San Francisco is usually mild and sunny, but it can swing from foggy 58°F mornings

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



*****TASK LIST*****

2: Write a detailed weather report for San Francisco for tomorrow, October 7, 2023.
3: Create a summary of the week's weather forecast for San Francisco.
4: Generate a comparison of today's weather in San Francisco with the weather in another major city.
5: Draft a weather advisory for outdoor activities in San Francisco based on today's conditions.
6: Prepare a list of recommended outdoor activities in San Francisco for a warm day like today.

*****NEXT TASK*****

2: Write a detailed weather report for San Francisco for tomorrow, October 7, 2023.
Thought: I need to gather current weather information for San Francisco for today, October 7, 2023, to create a detailed report.
Action: Search
Action Input: "San Francisco weather report October 7 2023"
Observation: ['Temperature (°F), Max, Average, Min. Max Temperature, 96, 88.43, 70. Avg Temperature, 77.25, 71.82, 62.85. Min Temperature, 63, 59.43, 54 ...', 'Daily Observations ; 10:56 PM, 58 °F · 55 °F ; 9:56 PM, 58 °

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: I need to gather current weather information for San Francisco for today, October 8, 2023, to create a detailed weather report. 
Action: Search
Action Input: "San Francisco weather report October 8, 2023"
Observation: ['This report shows the past weather for San Francisco, You can drill down from year to month and even day level reports by clicking on the graphs.', 'On Wednesday, inland areas throughout the region are expected to see temperatures up to 110, while coastal areas could reach up to 95 80s ...', 'Temperature (°F), Max, Average, Min. Max Temperature, 88, 77.25, 72. Avg Temperature, 71.5, 64.93, 60. Min Temperature, 64, 57.63, 54. Dew Point (°F), Max ...', 'Temperature (°F), Actual, Historic Avg. Record. High Temp, 76, 73, 95. Low Temp, 52, 53.8, 44. Day Average Temp, 61.75, 63.4, - ...', 'This report shows the past weather for Los Angeles / USC Campus Downtown, providing a weather history for October 2023. 75°F warm 85°F hot 95°F', "Tuesday is in line to be San Fran

{'objective': 'Write a weather report for SF today'}